# AI 시험 연습 문제 - 풀이

Time Series (LSTM/Conv1D/Encoder-Decoder) + Recommender System (NCF/NGCF) 핵심 코드 풀이

## Setup

In [1]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 핵심 개념 1. Sliding Window (슬라이딩 윈도우)

**목표**: 1차원 시계열 데이터를 딥러닝 모델이 학습할 수 있는 (입력 시퀀스, 다음 값) 쌍으로 변환.

- `sequence_length = 50` 이면, 과거 50일의 값을 입력 `X`로, 바로 다음 날의 값을 레이블 `y`로 사용.
- 전체 길이가 `N`인 시계열에서 `N - sequence_length` 개의 샘플 생성.
- 인덱스 `i`번째 샘플: `X = data[i:i+50]`, `y = data[i+50]`
- 입력 텐서 shape: `(num_samples, sequence_length, num_features)` → RNN/LSTM 표준 포맷

**왜 중요?**: 시계열은 "다음 값 예측"이 기본 과제이며, sliding window는 모든 시계열 딥러닝 파이프라인의 첫 단계.

### Problem 1. Sliding Window

In [ ]:
sequence_length = 50

def convert_data_into_tensors(data_seq):
    features, labels = [], []
    for i in range(len(data_seq) - sequence_length):
        features.append(### data_seq[i:i + sequence_length] ## Write Answer)
        labels.append(### data_seq[i + sequence_length, 0] ## Write Answer)
    features, labels = np.array(features), np.array(labels)

    features = torch.tensor(features, dtype=torch.float32)
    labels = torch.tensor(labels, dtype=torch.float32)
    return features, labels

## 핵심 개념 2. LSTM Model (LSTM 모델 구조)

**LSTM (Long Short-Term Memory)**: 장기 의존성 문제를 해결하기 위한 RNN 변형.

**PyTorch `nn.LSTM` 핵심 인자**:
- `input_size`: 각 시점의 feature 개수 (예: 가격만 쓰면 1)
- `hidden_size`: hidden state 차원
- `num_layers`: 쌓을 LSTM 층 수
- `batch_first=True`: 입력을 `(batch, seq_len, input_size)` 형태로 기대

**forward 출력**: `out, (h_n, c_n) = self.lstm(x)`
- `out`: 모든 time step의 hidden state `(batch, seq_len, hidden_size)`
- `h_n, c_n`: 마지막 hidden/cell state
- 예측에는 `out`의 마지막 time step이나 전체를 Linear에 통과시킴.

### Problem 2. LSTM Model

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.lstm = ### nn.LSTM(input_size, hidden_size, num_layers, batch_first=True) ## Write Answer
        self.linear = ### nn.Linear(hidden_size, 1) ## Write Answer

    def forward(self, x):
        out, _ = ### self.lstm(x) ## Write Answer
        return ### self.linear(out) ## Write Answer

## 핵심 개념 3. Training Loop (훈련 루프)

**PyTorch 훈련 5단계 (순서 고정)**:
1. `pred = model(batch_x)` — 순전파
2. `loss = loss_fn(pred, batch_y)` — 손실 계산
3. `optimizer.zero_grad()` — 이전 gradient 초기화 (누적 방지)
4. `loss.backward()` — 역전파로 gradient 계산
5. `optimizer.step()` — parameter 업데이트

**시계열 예측 tip**: LSTM 출력이 `(batch, seq_len, 1)`일 때, 마지막 time step만 사용하려면 `model(x)[:, -1, 0]`.

**`zero_grad` 순서 주의**: `backward()` 앞에 두어야 함. PyTorch는 gradient를 누적하므로 초기화가 필수.

### Problem 3. Training Loop

In [ ]:
def train_one_epoch(model, train_loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        pred = ### model(batch_x)[:, -1, 0] ## Write Answer
        loss = ### loss_fn(pred, batch_y) ## Write Answer

        ### optimizer.zero_grad() ## Write Answer
        ### loss.backward() ## Write Answer
        ### optimizer.step() ## Write Answer

        total_loss += loss.item()
    return total_loss / len(train_loader)

## 핵심 개념 4. Conv1D Model (1D 합성곱)

**`nn.Conv1d` 입력 형태**: `(batch, in_channels, seq_len)` — 채널이 두 번째 차원.

**시계열 데이터는 보통** `(batch, seq_len, features)` 형태이므로, **Conv1D 앞뒤로 `transpose(1, 2)` 필요**:
1. Conv 입력 전: `(batch, seq_len, features)` → `(batch, features, seq_len)` 로 transpose
2. Conv 실행
3. FC/다음 layer 앞: `(batch, out_channels, new_seq_len)` → `(batch, new_seq_len, out_channels)` 로 다시 transpose

**왜?**: `features`가 Conv의 채널에 해당하고, 커널은 시간 축(seq_len)을 따라 sliding 하기 때문.

### Problem 4. Conv1D Model

In [ ]:
class Conv1DModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(Conv1DModel, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=input_size,
                                out_channels=hidden_size,
                                kernel_size=2, stride=1)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = ### x.transpose(1, 2) ## Write Answer
        x = self.conv1d(x)
        x = ### x.transpose(1, 2) ## Write Answer
        return self.fc(x)

## 핵심 개념 5. Encoder-Decoder Architecture (Seq2Seq)

**구조**: 입력 시퀀스를 인코더가 하나의 hidden state `h`로 압축 → 디코더가 `h`로부터 타깃 시퀀스를 하나씩 생성.

**Forward 흐름**:
1. `h = self.encoder(source)` — 입력 전체를 읽고 context vector 반환
2. 초기 decoder 입력 = source의 **마지막 time step** `source[:, -1, :].unsqueeze(1)` (shape `(batch, 1, input_size)`)
3. `target_len` 만큼 loop:
   - `out, h = self.decoder(input, h)` — 한 step 생성
   - 생성된 `out`을 다음 step의 입력으로 재사용 (**autoregressive**)
4. `torch.stack(predictions, dim=1)` — time step list를 `(batch, target_len, dim)` 텐서로 합침.

**Autoregressive 생성의 핵심**: 이전 스텝 출력을 현재 스텝 입력으로 feedback → 다중 step 예측 가능.

### Problem 5. Encoder-Decoder Forward

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(EncoderRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

    def forward(self, x):
        _, h = self.rnn(x)
        return h

class DecoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(DecoderRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x, h):
        out, h = self.rnn(x, h)
        out = self.fc(out)
        return out, h

class RNNRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(RNNRNN, self).__init__()
        self.encoder = EncoderRNN(input_size, hidden_size, num_layers)
        self.decoder = DecoderRNN(input_size, hidden_size, num_layers)

    def forward(self, source, target_len):
        h = ### self.encoder(source) ## Write Answer
        predictions = []
        input = ### source[:, -1, :].unsqueeze(1) ## Write Answer
        for t in range(target_len):
            out, h = self.decoder(input, h)
            predictions.append(out.squeeze(1))
            input = ### out ## Write Answer
        outputs = ### torch.stack(predictions, dim=1) ## Write Answer
        return outputs

## 핵심 개념 6. Neural Collaborative Filtering (NCF)

**아이디어**: 유저/아이템을 잠재 벡터(embedding)로 학습하고, 두 벡터를 MLP에 통과시켜 rating 예측.

**구조**:
- `user_embedding`: `nn.Embedding(n_users, emb_dim=32)` — 유저 ID → 32차원 벡터
- `movie_embedding`: `nn.Embedding(n_movies, 32)`
- Forward에서 두 임베딩을 **`dim=1` 기준으로 concat** → 64차원
- FC → ReLU → FC(→ 1차원) 로 최종 평점 예측

**`nn.Embedding(num_embeddings, embedding_dim)`**: lookup table. 입력은 정수 인덱스 (유저/아이템 ID), 출력은 해당 row의 학습 가능한 벡터.

**dim=1 concat 이유**: `(batch, 32)` + `(batch, 32)` → `(batch, 64)`. dim=0은 배치 축이므로 써선 안 됨.

### Problem 6. NCF Model

In [ ]:
class Neural_Collaborative_Filtering(nn.Module):
    def __init__(self, n_users, n_movies):
        super().__init__()
        self.user_embedding = ### nn.Embedding(n_users, 32) ## Write Answer
        self.movie_embedding = ### nn.Embedding(n_movies, 32) ## Write Answer

        self.fc1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)

    def forward(self, users, movies, ratings=None):
        user_embedding = self.user_embedding(users)
        movie_embedding = self.movie_embedding(movies)
        input_embedding = ### torch.cat([user_embedding, movie_embedding], dim=1) ## Write Answer

        hidden_feature = self.fc1(input_embedding)
        hidden_feature = self.relu(hidden_feature)
        output = self.fc2(hidden_feature)
        return output

## 핵심 개념 7. Precision@k / Recall@k (추천 시스템 평가)

**정의** (threshold 이상을 "관련 있음(relevant)"으로 간주):
- **Precision@k** = (top-k 중 실제 relevant 개수) / k — 추천한 것 중 얼마나 맞았나
- **Recall@k** = (top-k 중 실제 relevant 개수) / (전체 relevant 개수) — 실제 relevant 중 얼마나 잡아냈나

**계산 순서**:
1. 유저별 (예측 rating, 실제 rating) 리스트를 **예측 rating 내림차순**으로 정렬
2. 전체에서 `true_r >= threshold` 개수 = `n_real_relevant`
3. 상위 k개 중 `true_r >= threshold` 개수 = `n_real_relevant_in_top_k`
4. Precision = `hits / k`, Recall = `hits / n_real_relevant` (분모 0이면 0 처리)

**주의**: `threshold`(예: 3.5) 이상이어야 "긍정 샘플". 아니면 이진 분류 문제가 성립 안 함.

### Problem 7. Precision@k & Recall@k

In [ ]:
def compute_precision_recall(user_est_true, k=10, threshold=3.5):
    precisions = dict()
    recalls = dict()
    for user_id, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        n_real_relevant = ### sum((true_r >= threshold) for (_, true_r) in user_ratings) ## Write Answer
        recommended_k = ### user_ratings[:k] ## Write Answer
        n_real_relevant_in_top_k = ### sum((true_r >= threshold) for (est, true_r) in recommended_k) ## Write Answer

        precisions[user_id] = ### n_real_relevant_in_top_k / k ## Write Answer
        if n_real_relevant:
            recalls[user_id] = ### n_real_relevant_in_top_k / n_real_relevant ## Write Answer
        else:
            recalls[user_id] = 0
    return precisions, recalls

## 핵심 개념 8. NGCF Normalization (그래프 정규화)

**NGCF (Neural Graph Collaborative Filtering)**: 유저-아이템 interaction을 **그래프(bipartite)**로 보고, GNN으로 embedding 전파.

**정규화 항**: 각 엣지 `(u, i)`의 메시지는 두 노드 degree의 기하평균의 역수로 가중.

$$ \text{norm}_{u,i} = \frac{1}{\sqrt{|\mathcal{N}_u|} \cdot \sqrt{|\mathcal{N}_i|}} = \frac{1}{\sqrt{\text{deg}(u) \cdot \text{deg}(i)}} $$

**왜?**: degree가 큰 인기 노드가 지나치게 영향을 주는 것을 방지(symmetric normalization, GCN에서 유래).

- `deg[src]`: 엣지별 src 노드의 degree를 pull
- `deg[dst]`: 엣지별 dst 노드의 degree를 pull
- `torch.sqrt(deg[src] * deg[dst])` 후 역수

### Problem 8. NGCF Layer: Normalization

In [ ]:
def compute_norm(deg, src, dst):
    norm = ### 1.0 / torch.sqrt(deg[src] * deg[dst]) ## Write Answer
    return norm

## 핵심 개념 9. NGCF Edge Message (엣지 메시지)

**NGCF의 핵심 수식** (src 노드 u 기준으로 이웃 i로부터 받는 메시지):

$$ m_{u \leftarrow i} = W_1 e_i + W_2 (e_i \odot e_u) $$

- `W1 * dst_feat`: 기본 feature 전달 (GCN 유사)
- `W2 * (dst_feat * src_feat)`: **element-wise 곱**으로 affinity(상호작용) 정보 인코딩 — NGCF의 핵심 기여점

이후 `norm` 곱해 degree 정규화 적용.

**양방향 그래프**이므로 dst 노드 i가 src u에게 받는 메시지도 `W1(src_feat) + W2(src_feat * dst_feat)` 로 대칭 계산.

**vs. GCN**: GCN은 `W * e_i`만 전파. NGCF는 `e_i * e_u` 항 추가로 유저-아이템 "친화도"를 더 잘 포착.

### Problem 9. NGCF Layer: Edge Messages

In [ ]:
class NGCFLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(input_dim, output_dim)
        self.W2 = nn.Linear(input_dim, output_dim)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, edge_index, node_features, user_num, item_num):
        src, dst = edge_index

        deg = torch.zeros(node_features.size(0), device=node_features.device)
        deg.index_add_(0, src, torch.ones_like(src, dtype=torch.float))
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))

        norm = 1.0 / torch.sqrt(deg[src] * deg[dst])

        src_feat = node_features[src]
        dst_feat = node_features[dst]

        edge_messages_for_src = ### self.W1(dst_feat) + self.W2(dst_feat * src_feat) ## Write Answer
        edge_messages_for_src *= norm.unsqueeze(1)

        edge_messages_for_dst = ### self.W1(src_feat) + self.W2(src_feat * dst_feat) ## Write Answer
        edge_messages_for_dst *= norm.unsqueeze(1)

        aggregated_messages = torch.zeros_like(node_features)
        aggregated_messages.index_add_(0, src, edge_messages_for_src)
        aggregated_messages[:user_num] += self.W1(node_features[:user_num])
        aggregated_messages.index_add_(0, dst, edge_messages_for_dst)
        aggregated_messages[user_num:] += self.W1(node_features[user_num:])

        return self.leaky_relu(aggregated_messages)

## 핵심 개념 10. NGCF Final Feature (계층별 concatenation)

**NGCF의 최종 representation**: layer 0 ~ layer L의 모든 embedding을 **feature 축(dim=-1)으로 concat**.

$$ e^{final} = e^{(0)} \parallel e^{(1)} \parallel \cdots \parallel e^{(L)} $$

- `layer_outputs`는 `[초기 embedding, layer1 out, layer2 out, ...]` 리스트
- `torch.concat(..., dim=-1)` 로 **column-wise** 연결 → 최종 차원 = sum of layer dims
- 상위 `num_users` row → 유저 feature, 하위 → 아이템 feature

**왜 concat?**: 서로 다른 hop의 이웃 정보(local vs. distant)를 모두 보존. 평균이나 sum은 정보 손실 가능.

### Problem 10. NGCF: Final Features

In [ ]:
def build_final_features(layer_outputs, num_users):
    final_features = ### torch.concat(layer_outputs, dim=-1) ## Write Answer
    user_features = final_features[:num_users]
    item_features = final_features[num_users:]
    return user_features, item_features

## 핵심 개념 11. BPR Loss (Bayesian Personalized Ranking)

**BPR 아이디어**: 명시적 rating이 아닌 **implicit feedback** 상황에서, 유저가 상호작용한 아이템(positive)의 점수가 상호작용하지 않은 아이템(negative)보다 높아야 한다는 pairwise 랭킹 손실.

**수식**:
$$ \mathcal{L}_{BPR} = - \frac{1}{N} \sum_{(u, i^+, i^-)} \log \sigma(s_{u, i^+} - s_{u, i^-}) $$

- `pos_scores = sum(user_emb * pos_item_emb, dim=1)` — 유저-positive 아이템 내적
- `neg_scores = sum(user_emb * neg_item_emb, dim=1)` — 유저-negative 아이템 내적
- `-mean(logsigmoid(pos - neg))`: pos가 neg보다 클수록 (차이가 클수록) loss 감소
- `reg_loss`: L2 정규화로 임베딩 크기 폭주 방지

**핵심 포인트**: `dim=1`은 feature 차원(배치별 내적), `F.logsigmoid`는 수치 안정적인 `log(sigmoid(x))`.

### Problem 11. BPR Loss

In [ ]:
def bpr_loss(user_emb, pos_item_emb, neg_item_emb, reg_weight=1e-4):
    pos_scores = ### torch.sum(user_emb * pos_item_emb, dim=1) ## Write Answer
    neg_scores = ### torch.sum(user_emb * neg_item_emb, dim=1) ## Write Answer
    loss = ### -torch.mean(F.logsigmoid(pos_scores - neg_scores)) ## Write Answer
    reg_loss = reg_weight * (user_emb.norm(2).pow(2) + pos_item_emb.norm(2).pow(2) + neg_item_emb.norm(2).pow(2)) / user_emb.size(0)
    return loss + reg_loss